In [2]:
!pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 2.9 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [3]:
# Required libraries
import ast
from pathlib import Path

import pandas as pd
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
DATA_DIR = Path.cwd()
MOVIES_FILE = DATA_DIR / "tmdb_5000_movies.csv"
CREDITS_FILE = DATA_DIR / "tmdb_5000_credits.csv"

if not MOVIES_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {MOVIES_FILE.name}. Put it in {DATA_DIR} or update DATA_DIR."
    )
if not CREDITS_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {CREDITS_FILE.name}. Put it in {DATA_DIR} or update DATA_DIR."
    )

movie = pd.read_csv(MOVIES_FILE)
credit = pd.read_csv(CREDITS_FILE)

print(f"Movies: {movie.shape}")
print(f"Credits: {credit.shape}")

Movies: (4803, 20)
Credits: (4803, 4)


In [5]:
# Merge using the movie title, then keep only the columns needed for the recommender.
# reset_index() is important: similarity-matrix positions must match dataframe positions.
data = movie.merge(credit, on="title", how="inner")
data = data[[
    "genres", "id", "keywords", "overview",
    "release_date", "title", "cast", "crew"
]].copy()

data.dropna(subset=["genres", "keywords", "overview", "release_date", "title", "cast", "crew"], inplace=True)
data.reset_index(drop=True, inplace=True)

print(f"Prepared dataset: {data.shape}")

Prepared dataset: (4805, 8)


In [6]:
def parse_json_list(value):
    """Return a parsed list from a TMDB JSON-like string."""
    if isinstance(value, list):
        return value
    if pd.isna(value) or str(value).strip() == "":
        return []
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []


def extract_names(value):
    """Extract the 'name' field from a TMDB list of dictionaries."""
    result = []
    for item in parse_json_list(value):
        if isinstance(item, dict) and item.get("name"):
            result.append(str(item["name"]))
    return result


def extract_top_cast(value, n=3):
    """Extract the first n cast member names."""
    result = []
    for item in parse_json_list(value)[:n]:
        if isinstance(item, dict) and item.get("name"):
            result.append(str(item["name"]))
    return result


def extract_directors(value):
    """Extract director names from the crew list."""
    result = []
    for item in parse_json_list(value):
        if isinstance(item, dict) and item.get("job") == "Director" and item.get("name"):
            result.append(str(item["name"]))
    return result


data["genres"] = data["genres"].apply(extract_names)
data["keywords"] = data["keywords"].apply(extract_names)
data["cast"] = data["cast"].apply(extract_top_cast)
data["crew"] = data["crew"].apply(extract_directors)
data["overview"] = data["overview"].astype(str).str.split()


In [7]:
for column in ["genres", "keywords", "cast", "crew"]:
    data[column] = data[column].apply(
        lambda values: [str(value).replace(" ", "") for value in values]
    )

data["tags"] = (
    data["genres"]
    + data["keywords"]
    + data["overview"]
    + data["cast"]
    + data["crew"]
).apply(lambda values: " ".join(values).lower())

data["title_normalized"] = data["title"].astype(str).str.strip().str.lower()
data = data[["id", "title", "title_normalized", "tags"]].reset_index(drop=True)

data.head()


,id,title,title_normalized,tags
0,19995,Avatar,avatar,action adventure fantasy sciencefiction cultur...
1,285,Pirates of the Caribbean: At World's End,pirates of the caribbean: at world's end,adventure fantasy action ocean drugabuse exoti...
2,206647,Spectre,spectre,action adventure crime spy basedonnovel secret...
3,49026,The Dark Knight Rises,the dark knight rises,action crime drama thriller dccomics crimefigh...
4,49529,John Carter,john carter,action adventure sciencefiction basedonnovel m...


In [8]:
ps = PorterStemmer()

def stemming(text):
    return " ".join(ps.stem(word) for word in str(text).split())

data["tags"] = data["tags"].apply(stemming)

cv = CountVectorizer(max_features=5000, stop_words="english")
vectors = cv.fit_transform(data["tags"])

print(f"Vector matrix shape: {vectors.shape}")


Vector matrix shape: (4805, 5000)


In [9]:
similarity = cosine_similarity(vectors)
print(f"Similarity matrix shape: {similarity.shape}")


Similarity matrix shape: (4805, 4805)


In [10]:
def recommend(movie_title, n=5):
    """Return up to n movies similar to the supplied movie title."""
    if not isinstance(movie_title, str) or not movie_title.strip():
        raise ValueError("movie_title must be a non-empty string.")

    query = movie_title.strip().lower()
    matches = data.index[data["title_normalized"] == query].tolist()

    if not matches:
        partial = data[data["title_normalized"].str.contains(query, regex=False, na=False)]
        if partial.empty:
            print(f"Movie not found: {movie_title}")
            print("Try one of these titles:")
            print(data["title"].head(10).to_string(index=False))
            return pd.DataFrame(columns=["title", "id", "similarity"])
        movie_index = partial.index[0]
        print(f"Using: {data.iloc[movie_index]['title']}")
    else:
        movie_index = matches[0]

    scores = similarity[movie_index]
    ranked_indices = scores.argsort()[::-1]
    ranked_indices = [idx for idx in ranked_indices if idx != movie_index][:n]

    recommendations = data.iloc[ranked_indices][["title", "id"]].copy()
    recommendations["similarity"] = [float(scores[idx]) for idx in ranked_indices]
    recommendations.reset_index(drop=True, inplace=True)
    return recommendations


In [11]:
recommendations = recommend("Avatar", n=5)
recommendations


,title,id,similarity
0,Aliens vs Predator: Requiem,440,0.286770
1,Aliens,679,0.269014
2,Falcon Rising,270938,0.260513
3,Independence Day,602,0.255609
4,Titan A.E.,7450,0.250387


In [12]:
if not recommendations.empty:
    for title in recommendations["title"]:
        print(title)


Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.
